# SmolVLA-Training für VLA_Trial (Google Colab, kostenlos)

Trainiert SmolVLA auf dem Datensatz `HF_USER/so101_blocks_into_box` und lädt das Modell
privat nach `HF_USER/smolvla_so101_blocks_into_box` hoch.

**Vorbereitung (einmalig):**
1. Menü **Laufzeit → Laufzeittyp ändern → T4 GPU** → Speichern.
2. Links auf das **Schlüssel-Symbol** (Secrets) → neues Secret `HF_TOKEN` mit deinem
   Hugging-Face-Token (Write-Rechte) → „Notebook-Zugriff“ einschalten.
3. Unten `HF_USER` eintragen.

**Ablauf:** Zellen 1–4 ausführen, dann Zelle 5 (Training starten).
Alle `SAVE_FREQ` Schritte wird ein Checkpoint auf Hugging Face gesichert. Bricht die
Colab-Sitzung ab (kostenlose Sitzungen werden nach einigen Stunden getrennt), später
Zellen 1–4 erneut ausführen und dann **Zelle 6** (Fortsetzen) statt Zelle 5.

## 1. Einstellungen

In [ ]:
HF_USER = "JRrobots4all"  # dein Hugging-Face-Name
DATASET = "so101_blocks_into_box"
MODEL = "smolvla_so101_blocks_into_box"
STEPS = 20000
BATCH = 32  # passt auf eine T4 (16 GB); bei "CUDA out of memory" auf 16 senken
SAVE_FREQ = 2000  # Checkpoint-Abstand in Schritten, wird jeweils auf Hugging Face gesichert
LEROBOT_REV = "3f797de5952432284a7c77ad2760218cb74e2f26"  # gleiche LeRobot-Version wie auf dem Laptop

import os

assert HF_USER != "CHANGE_ME", "Bitte oben HF_USER eintragen."
os.environ.update(HF_USER=HF_USER, DATASET=DATASET, MODEL=MODEL, STEPS=str(STEPS),
                  BATCH=str(BATCH), SAVE_FREQ=str(SAVE_FREQ), LEROBOT_REV=LEROBOT_REV)

## 2. GPU prüfen
Hier muss eine NVIDIA-GPU (z. B. Tesla T4) erscheinen, sonst Laufzeittyp ändern.

In [ ]:
!nvidia-smi

## 3. LeRobot installieren (dauert einige Minuten)

In [ ]:
%%bash
set -e
cd /content
[ -d lerobot ] || git clone -q https://github.com/huggingface/lerobot.git
cd lerobot
git fetch -q origin
git checkout -q "$LEROBOT_REV"
pip install -q -e ".[training,smolvla]"
echo "LeRobot installiert ($(git rev-parse --short HEAD))"

## 4. Bei Hugging Face anmelden

In [ ]:
try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:  # kein Secret hinterlegt: interaktiv anmelden
    print("Kein HF_TOKEN-Secret gefunden:", e)
    from huggingface_hub import notebook_login

    notebook_login()
!hf auth whoami

## 5. Training starten (erste Sitzung)
Fortschritt erscheint unten als Log (`step`, `loss`). Der Loss sollte über die Zeit sinken.

In [ ]:
%%bash
cd /content/lerobot
lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id="$HF_USER/$DATASET" \
  --policy.repo_id="$HF_USER/$MODEL" \
  --policy.private=true \
  --rename_map='{"observation.images.top": "observation.images.camera1", "observation.images.side": "observation.images.camera2"}' \
  --batch_size="$BATCH" \
  --steps="$STEPS" \
  --save_freq="$SAVE_FREQ" \
  --save_checkpoint_to_hub=true \
  --num_workers=2 \
  --output_dir="/content/outputs/$MODEL" \
  --job_name="$MODEL" \
  --policy.device=cuda

## 6. Training fortsetzen (nach einem Abbruch)
Lädt den letzten Checkpoint von Hugging Face und trainiert mit denselben Einstellungen weiter
(Optimizer, Schrittzähler und Datenreihenfolge werden wiederhergestellt).

In [ ]:
%%bash
cd /content/lerobot
lerobot-train --config_path="$HF_USER/$MODEL" --resume=true

## Fertig
Nach dem letzten Schritt liegt das Modell unter `https://huggingface.co/JRrobots4all/smolvla_so101_blocks_into_box`.
Auf dem Laptop dann `scripts\rollout.bat green` bzw. `blue`.
Danach die Laufzeit trennen (**Laufzeit → Laufzeit trennen und löschen**), damit das GPU-Kontingent nicht verbraucht wird.